# 06 — Controlled Comparison: Baseline vs V1-Fixed

**Purpose**: Compare the base model and fine-tuned model with shared
post-processing and evaluation, isolating the effect of fine-tuning.

**Design**:
- Both models use the same stop tokens, post-processing, and evaluation.
- Each model uses its own best generation config:
  - **Base model**: no `repetition_penalty` (penalty hurts it — pushes the model
    toward exotic SQL constructs like JOINs and subqueries)
  - **V1-fixed**: `repetition_penalty=1.3` (as used in original evaluation)
- The only prompt-level difference: v1-fixed prepends the Llama-3 chat template
  prefix, because SFTTrainer applied it during training.
- Reports results **with and without** `clean_wikisql()` for both models.

**Run on**: Google Colab (T4 GPU)

In [2]:
!pip install -q "datasets<3.0" transformers peft bitsandbytes accelerate torch

In [4]:
from huggingface_hub import login
login()

## 1. Shared Setup

In [5]:
import torch
import re
import json
import sqlite3
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load test data
ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
examples = list(ds['test'].select(range(500)))
print(f"Loaded {len(examples)} test examples")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

Loaded 500 test examples


In [6]:
# ── Generation configs ─────────────────────────────────────────────────────
# Shared: same stop tokens, same max_new_tokens, same do_sample=False
# Different: repetition_penalty (hurts base model, helps/neutral for v1-fixed)

STOP_TOKEN_IDS = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

# Base model config: NO repetition_penalty
# (The base model generates diverse, complex SQL. Repetition penalty makes it
#  worse by penalizing common SQL tokens like FROM/WHERE/AND, pushing it toward
#  even more exotic constructs.)
BASE_GENERATION_CONFIG = dict(
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=STOP_TOKEN_IDS,
)

# V1-fixed config: WITH repetition_penalty=1.3 (as in original evaluation)
V1F_GENERATION_CONFIG = dict(
    max_new_tokens=128,
    do_sample=False,
    repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=STOP_TOKEN_IDS,
)

# Chat template prefix — only for v1-fixed (matches its training format)
CHAT_PREFIX = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"

print("Base model config:")
for k, v in BASE_GENERATION_CONFIG.items():
    print(f"  {k}: {v}")
print("\nV1-fixed config:")
for k, v in V1F_GENERATION_CONFIG.items():
    print(f"  {k}: {v}")

Base model config:
  max_new_tokens: 128
  do_sample: False
  pad_token_id: 128009
  eos_token_id: [128009, 128009]

V1-fixed config:
  max_new_tokens: 128
  do_sample: False
  repetition_penalty: 1.3
  pad_token_id: 128009
  eos_token_id: [128009, 128009]


In [7]:
# ── Shared post-processing (identical for both models) ─────────────────────

def fix_column_names(sql: str, columns: list) -> str:
    """Wrap column name references in backticks, handling underscore variants."""
    for col in sorted(columns, key=len, reverse=True):
        if not col:
            continue
        quoted = f'`{col}`'
        underscore_variant = re.sub(r'[\s\-–]+', '_', col)
        variants = [col]
        if underscore_variant.lower() != col.lower():
            variants.append(underscore_variant)
        for variant in variants:
            if re.search(re.escape(quoted), sql, re.IGNORECASE):
                break
            if re.search(re.escape(variant), sql, re.IGNORECASE):
                sql = re.sub(re.escape(variant), quoted, sql, flags=re.IGNORECASE)
                break
    return sql


def clean_wikisql(sql: str) -> str:
    """Strip SQL constructs not supported by WikiSQL grammar."""
    sql = re.sub(r'\s+OR\s+.+$',              '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r'\s+ORDER\s+BY\s+.+$',      '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r'\s+LIMIT\s+.+$',           '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"\s+AND\s+`[^`]+`\s+IS\s+NOT\s+NULL", '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"""\s+AND\s+`[^`]+`\s*!=\s*['"]?['"]?""", '', sql, flags=re.IGNORECASE).strip()
    return sql


def extract_sql(response: str, columns: list) -> str:
    """Extract and clean SQL from raw model output."""
    if "```" in response:
        lines = response.split('\n')
        response = ' '.join(l for l in lines if not l.strip().startswith('```')).strip()

    for line in response.split('\n'):
        line = line.strip()
        line = re.sub(r'\bassistant\b', '', line, flags=re.IGNORECASE).strip()
        if line.upper().startswith('SELECT'):
            line = line.split(';')[0].strip()
            if '--' in line: line = line[:line.index('--')].strip()
            if '###' in line: line = line[:line.index('###')].strip()
            line = re.sub(r'\s+LIMIT\s+.*$', '', line, flags=re.IGNORECASE).strip()
            line = re.sub(r'\bFROM\s+\w+\b', 'FROM table', line, flags=re.IGNORECASE)
            line = fix_column_names(line, columns)
            line = re.sub(r"\s+AND\s+`[^`]+`\s*=\s*''", '', line, flags=re.IGNORECASE)
            return line

    return response.split('\n')[0].split(';')[0].strip()


print("Post-processing functions defined (shared by both models)")

Post-processing functions defined (shared by both models)


In [8]:
# ── Generation function ───────────────────────────────────────────────────

def generate_sql(model, question, columns, types,
                 use_chat_prefix=False, gen_config=None):
    """Generate SQL with the specified generation config.

    Args:
        model: loaded model (base or with adapter)
        question: natural language question
        columns: table column names
        types: table column types
        use_chat_prefix: prepend Llama-3 chat template prefix
        gen_config: generation parameters dict
    """
    col_defs = ", ".join(f"{name} ({dtype})" for name, dtype in zip(columns, types))
    raw_prompt = (
        f"### Input:\n"
        f"Columns: {col_defs}\n\n"
        f"Question: {question}\n\n"
        f"### SQL:\n"
    )

    if use_chat_prefix:
        full_prompt = CHAT_PREFIX + raw_prompt
        inputs = tokenizer(full_prompt, return_tensors="pt",
                           add_special_tokens=False).to(model.device)
    else:
        full_prompt = raw_prompt
        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_config)

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_sql(response, columns)


print("generate_sql() defined")
print("  Base model:  gen_config=BASE_GENERATION_CONFIG, use_chat_prefix=False")
print("  V1-fixed:    gen_config=V1F_GENERATION_CONFIG,  use_chat_prefix=True")

generate_sql() defined
  Base model:  gen_config=BASE_GENERATION_CONFIG, use_chat_prefix=False
  V1-fixed:    gen_config=V1F_GENERATION_CONFIG,  use_chat_prefix=True


In [9]:
# ── Evaluation functions ───────────────────────────────────────────────────

AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<"]

def build_sql_string(sql, columns, types):
    agg = AGG_OPS[sql["agg"]]
    sel_col = columns[sql["sel"]]
    select_clause = f"SELECT {agg}(`{sel_col}`)" if agg else f"SELECT `{sel_col}`"
    where_clauses = []
    for col_idx, op_idx, cond in zip(
        sql["conds"]["column_index"],
        sql["conds"]["operator_index"],
        sql["conds"]["condition"],
    ):
        col = columns[col_idx]
        op = COND_OPS[op_idx]
        col_type = types[col_idx]
        if col_type in ("real", "number"):
            cleaned = str(cond).replace(",", "")
            try:
                float(cleaned)
                where_clauses.append(f"`{col}` {op} {cleaned}")
            except:
                where_clauses.append(f"`{col}` {op} '{str(cond).replace(chr(39), chr(39)*2)}'")
        else:
            escaped = str(cond).replace("'", "''")
            where_clauses.append(f"`{col}` {op} '{escaped}'")
    where_clause = " WHERE " + " AND ".join(where_clauses) if where_clauses else ""
    return select_clause + " FROM table" + where_clause


def build_db(table):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    type_map = {"text": "TEXT", "number": "REAL", "real": "REAL"}
    col_defs = ", ".join(
        f'`{col}` {type_map.get(t, "TEXT")}'
        for col, t in zip(table["header"], table["types"])
    )
    cursor.execute(f"CREATE TABLE data ({col_defs})")
    for row in table["rows"]:
        converted = []
        for val, col_type in zip(row, table["types"]):
            if col_type in ("real", "number"):
                try:
                    converted.append(float(str(val).replace(",", "")))
                except:
                    converted.append(val)
            else:
                converted.append(val)
        placeholders = ", ".join(["?"] * len(converted))
        cursor.execute(f"INSERT INTO data VALUES ({placeholders})", converted)
    conn.commit()
    return conn


def execute_sql(table, sql):
    sql = sql.replace("FROM table", "FROM data")
    try:
        conn = build_db(table)
        cursor = conn.cursor()
        cursor.execute(sql)
        results = cursor.fetchall()
        conn.close()
        return results, None
    except Exception as e:
        return [], str(e)


def normalize_result(result):
    return sorted([str(row) for row in result])


def evaluate(predictions, examples):
    """Compute execution accuracy, syntax error rate, and wrong result rate."""
    total = len(predictions)
    correct = syntax_errors = wrong_result = 0
    for pred, ex in zip(predictions, examples):
        table = ex["table"]
        gold_sql = build_sql_string(ex["sql"], table["header"], table["types"])
        pred_results, pred_error = execute_sql(table, pred)
        gold_results, _ = execute_sql(table, gold_sql)
        if pred_error:
            syntax_errors += 1
        elif normalize_result(pred_results) == normalize_result(gold_results):
            correct += 1
        else:
            wrong_result += 1
    return {
        "exec_acc": correct / total,
        "syntax_err": syntax_errors / total,
        "wrong_result": wrong_result / total,
        "correct": correct,
        "syntax_errors": syntax_errors,
        "wrong_results": wrong_result,
        "total": total,
    }


print("Evaluation functions defined")

Evaluation functions defined


## 2. Evaluate Base Model

In [10]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
base_model.eval()
print(f"Base model loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Base model loaded. Memory: 6.1 GB


In [11]:
# Sanity check
print("=== Base Model Sanity Check (no repetition_penalty) ===")
for i in range(3):
    ex = examples[i]
    pred = generate_sql(base_model, ex['question'],
                        ex['table']['header'], ex['table']['types'],
                        use_chat_prefix=False,
                        gen_config=BASE_GENERATION_CONFIG)
    print(f"Q:    {ex['question']}")
    print(f"Pred: {pred}")
    print(f"Gold: {ex['sql']['human_readable']}")
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Base Model Sanity Check (no repetition_penalty) ===
Q:    What is terrence ross' nationality
Pred: SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
Gold: SELECT Nationality FROM table WHERE Player = Terrence Ross

Q:    What clu was in toronto 1995-96
Pred: SELECT * FROM table WHERE `School/Club Team` = 'Toronto' AND `Years in Toronto` = '1995-96'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 1995-96

Q:    which club was in toronto 2003-06
Pred: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 2003-06



In [12]:
base_preds = []
for ex in tqdm(examples, desc="Base model"):
    pred = generate_sql(base_model, ex['question'],
                        ex['table']['header'], ex['table']['types'],
                        use_chat_prefix=False,
                        gen_config=BASE_GENERATION_CONFIG)
    base_preds.append(pred)

print(f"Generated {len(base_preds)} base model predictions")

Base model:   0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 base model predictions


In [13]:
# Quick check before moving on
base_quick = evaluate(base_preds, examples)
print(f"Base model results:")
print(f"  Execution accuracy: {base_quick['exec_acc']:.1%}")
print(f"  Syntax error rate:  {base_quick['syntax_err']:.1%}")
print(f"  Wrong result rate:  {base_quick['wrong_result']:.1%}")

Base model results:
  Execution accuracy: 37.2%
  Syntax error rate:  21.8%
  Wrong result rate:  41.0%


In [14]:
# Save base predictions immediately (in case session is interrupted)
from google.colab import files

base_save = {
    "model": MODEL_ID,
    "adapter": None,
    "prompt_format": "raw",
    "generation_config": {"max_new_tokens": 128, "do_sample": False,
                          "repetition_penalty": None,
                          "stop_tokens": ["eos_token", "<|eot_id|>"]},
    "n_examples": len(base_preds),
    "exec_acc": base_quick["exec_acc"],
    "syntax_err": base_quick["syntax_err"],
    "predictions": base_preds,
}
with open("base_model_no_rp.json", "w") as f:
    json.dump(base_save, f, indent=2)
files.download("base_model_no_rp.json")
print("Base predictions saved.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Base predictions saved.


In [15]:
# Free GPU memory
del base_model
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"Memory freed. Current usage: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Memory freed. Current usage: 0.0 GB


## 3. Evaluate V1-Fixed Model

In [16]:
ADAPTER_ID = "oLittle-five/llama3-8b-wikisql-qlora"

ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_ID)
ft_model.eval()
print(f"V1 adapter loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

V1 adapter loaded. Memory: 6.2 GB


In [17]:
# Sanity check
print("=== V1-Fixed Sanity Check (with repetition_penalty=1.3) ===")
for i in range(3):
    ex = examples[i]
    pred = generate_sql(ft_model, ex['question'],
                        ex['table']['header'], ex['table']['types'],
                        use_chat_prefix=True,
                        gen_config=V1F_GENERATION_CONFIG)
    print(f"Q:    {ex['question']}")
    print(f"Pred: {pred}")
    print(f"Gold: {ex['sql']['human_readable']}")
    print()

=== V1-Fixed Sanity Check (with repetition_penalty=1.3) ===
Q:    What is terrence ross' nationality
Pred: SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
Gold: SELECT Nationality FROM table WHERE Player = Terrence Ross

Q:    What clu was in toronto 1995-96
Pred: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '1995-96'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 1995-96

Q:    which club was in toronto 2003-06
Pred: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 2003-06



In [18]:
v1f_preds = []
for ex in tqdm(examples, desc="V1-fixed"):
    pred = generate_sql(ft_model, ex['question'],
                        ex['table']['header'], ex['table']['types'],
                        use_chat_prefix=True,
                        gen_config=V1F_GENERATION_CONFIG)
    v1f_preds.append(pred)

print(f"Generated {len(v1f_preds)} v1-fixed predictions")

V1-fixed:   0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 v1-fixed predictions


In [19]:
# Quick check
v1f_quick = evaluate(v1f_preds, examples)
print(f"V1-fixed results:")
print(f"  Execution accuracy: {v1f_quick['exec_acc']:.1%}")
print(f"  Syntax error rate:  {v1f_quick['syntax_err']:.1%}")
print(f"  Wrong result rate:  {v1f_quick['wrong_result']:.1%}")

V1-fixed results:
  Execution accuracy: 51.8%
  Syntax error rate:  5.4%
  Wrong result rate:  42.8%


In [20]:
# Save v1-fixed predictions immediately
v1f_save = {
    "model": MODEL_ID,
    "adapter": ADAPTER_ID,
    "prompt_format": "chat_template_prefix",
    "generation_config": {"max_new_tokens": 128, "do_sample": False,
                          "repetition_penalty": 1.3,
                          "stop_tokens": ["eos_token", "<|eot_id|>"]},
    "n_examples": len(v1f_preds),
    "exec_acc": v1f_quick["exec_acc"],
    "syntax_err": v1f_quick["syntax_err"],
    "predictions": v1f_preds,
}
with open("v1_fixed_controlled.json", "w") as f:
    json.dump(v1f_save, f, indent=2)
files.download("v1_fixed_controlled.json")
print("V1-fixed predictions saved.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

V1-fixed predictions saved.


## 4. Full Results + Ablation

In [21]:
# ── Evaluate with and without clean_wikisql ───────────────────────────────
base_raw  = evaluate(base_preds, examples)
v1f_raw   = evaluate(v1f_preds, examples)

base_cleaned = [clean_wikisql(p) for p in base_preds]
v1f_cleaned  = [clean_wikisql(p) for p in v1f_preds]

base_clean = evaluate(base_cleaned, examples)
v1f_clean  = evaluate(v1f_cleaned, examples)

base_modified = sum(1 for a, b in zip(base_preds, base_cleaned) if a != b)
v1f_modified  = sum(1 for a, b in zip(v1f_preds, v1f_cleaned) if a != b)

# ── Display ────────────────────────────────────────────────────────────────
print("=" * 80)
print("CONTROLLED COMPARISON")
print("  Shared: stop_tokens, post-processing, evaluation")
print("  Base model: no repetition_penalty | V1-fixed: repetition_penalty=1.3")
print("=" * 80)

print(f"\n{'Config':<50} {'Exec Acc':>10} {'Syntax Err':>12} {'Wrong':>8}")
print("-" * 80)
print(f"{'Base (no clean_wikisql)':<50} {base_raw['exec_acc']:>10.1%} {base_raw['syntax_err']:>12.1%} {base_raw['wrong_result']:>8.1%}")
print(f"{'Base (with clean_wikisql)':<50} {base_clean['exec_acc']:>10.1%} {base_clean['syntax_err']:>12.1%} {base_clean['wrong_result']:>8.1%}")
print(f"{'V1-fixed (no clean_wikisql)':<50} {v1f_raw['exec_acc']:>10.1%} {v1f_raw['syntax_err']:>12.1%} {v1f_raw['wrong_result']:>8.1%}")
print(f"{'V1-fixed (with clean_wikisql)':<50} {v1f_clean['exec_acc']:>10.1%} {v1f_clean['syntax_err']:>12.1%} {v1f_clean['wrong_result']:>8.1%}")

print(f"\n--- Ablation: clean_wikisql effect ---")
print(f"Predictions modified — Base: {base_modified}/500, V1-fixed: {v1f_modified}/500")
print(f"Exec acc change — Base: {base_clean['exec_acc']-base_raw['exec_acc']:+.1%}, "
      f"V1-fixed: {v1f_clean['exec_acc']-v1f_raw['exec_acc']:+.1%}")

print(f"\n--- Improvement from fine-tuning ---")
print(f"Without clean_wikisql: {base_raw['exec_acc']:.1%} → {v1f_raw['exec_acc']:.1%} "
      f"({v1f_raw['exec_acc']-base_raw['exec_acc']:+.1%})")
print(f"With clean_wikisql:    {base_clean['exec_acc']:.1%} → {v1f_clean['exec_acc']:.1%} "
      f"({v1f_clean['exec_acc']-base_clean['exec_acc']:+.1%})")

CONTROLLED COMPARISON
  Shared: stop_tokens, post-processing, evaluation
  Base model: no repetition_penalty | V1-fixed: repetition_penalty=1.3

Config                                               Exec Acc   Syntax Err    Wrong
--------------------------------------------------------------------------------
Base (no clean_wikisql)                                 37.2%        21.8%    41.0%
Base (with clean_wikisql)                               36.8%        22.6%    40.6%
V1-fixed (no clean_wikisql)                             51.8%         5.4%    42.8%
V1-fixed (with clean_wikisql)                           51.4%         6.0%    42.6%

--- Ablation: clean_wikisql effect ---
Predictions modified — Base: 25/500, V1-fixed: 4/500
Exec acc change — Base: -0.4%, V1-fixed: -0.4%

--- Improvement from fine-tuning ---
Without clean_wikisql: 37.2% → 51.8% (+14.6%)
With clean_wikisql:    36.8% → 51.4% (+14.6%)


In [22]:
# ── Save full results ─────────────────────────────────────────────────────
controlled_results = {
    "experiment": "controlled_comparison",
    "description": (
        "Both models use same stop tokens (eos + eot_id), same post-processing, "
        "same evaluation. Base model uses no repetition_penalty (it degrades its "
        "output quality). V1-fixed uses repetition_penalty=1.3 (as in original eval). "
        "Only prompt difference: v1-fixed prepends chat template prefix to match training."
    ),
    "base_model": {
        "model": MODEL_ID, "adapter": None,
        "prompt_format": "raw",
        "repetition_penalty": None,
        "without_clean_wikisql": {"exec_acc": base_raw["exec_acc"], "syntax_err": base_raw["syntax_err"], "wrong_result": base_raw["wrong_result"]},
        "with_clean_wikisql":    {"exec_acc": base_clean["exec_acc"], "syntax_err": base_clean["syntax_err"], "wrong_result": base_clean["wrong_result"]},
        "predictions_modified_by_clean_wikisql": base_modified,
        "predictions": base_preds,
    },
    "v1_fixed": {
        "model": MODEL_ID, "adapter": ADAPTER_ID,
        "prompt_format": "chat_template_prefix",
        "repetition_penalty": 1.3,
        "without_clean_wikisql": {"exec_acc": v1f_raw["exec_acc"], "syntax_err": v1f_raw["syntax_err"], "wrong_result": v1f_raw["wrong_result"]},
        "with_clean_wikisql":    {"exec_acc": v1f_clean["exec_acc"], "syntax_err": v1f_clean["syntax_err"], "wrong_result": v1f_clean["wrong_result"]},
        "predictions_modified_by_clean_wikisql": v1f_modified,
        "predictions": v1f_preds,
    },
    "n_examples": 500,
}

with open("controlled_comparison_results.json", "w") as f:
    json.dump(controlled_results, f, indent=2)
files.download("controlled_comparison_results.json")
print("Full results saved and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Full results saved and downloaded!


## 5. Summary

This controlled comparison uses:
- **Same** stop tokens, post-processing (`extract_sql`, `clean_wikisql`), evaluation
- **Different** `repetition_penalty` per model (each uses its best config)
- **Different** prompt prefix for v1-fixed (required by its training format)

The `repetition_penalty` difference is documented and justified:
it degrades base model output quality (pushing toward exotic SQL) while having
negligible effect on the fine-tuned model's already-concise outputs.

**After running**, place these files in `results/`:
- `base_model_no_rp.json` — base predictions (no repetition penalty)
- `v1_fixed_controlled.json` — v1-fixed predictions
- `controlled_comparison_results.json` — full comparison with ablation

You also have `base_model_controlled.json` from the earlier run (base model
WITH repetition penalty, 10.2% accuracy) which demonstrates why the penalty
is harmful to the base model.